# 36 - Analyze suffix sensitivity and tapered P&P

This zero-GPU notebook analyzes the completed four-worker experiment `pi05-source-suffix-sensitivity-v1`. It requires the exact **220 identities × 3 arms** cohort. Every reported primary SR and SR delta uses all exact-matched episodes.

The three arms are: stock/no-op diagnostic baseline, ordinary full P&P, and decaying-tail P&P. Window sweeps are exploratory: outside the window the baseline outcome is retained; inside it the matching refinement-arm outcome is substituted.

## 1. Setup

In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Fetch and validate the exact three-arm cohort

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm

from analysis.suffix_sensitivity import (
    ARMS, SCORE_COLUMNS, apply_window_by_suite, bootstrap_rank_auc,
    failure_auc_table, load_artifact_tables, pair_arms, sensitivity_quartiles,
    summarize_all_arms, summarize_pair, threshold_sweep, top_windows,
    validate_cohort, window_sweep)
from pnp.config import Method
from pnp.diversity import (DIVERSITY_PAIR_KEYS,
    SOURCE_SUFFIX_SENSITIVITY_EXPERIMENT)
from pnp.store import SupabaseStore

EXPECTED_IDENTITIES = 220
REQUIRE_FULL_COHORT = True
REFRESH_ARTIFACTS = False
GRID_SIZE = 25
MIN_SELECTED = 10
OUTPUT = Path('suffix_sensitivity_outputs')
OUTPUT.mkdir(exist_ok=True)
store = SupabaseStore()

rows = pd.DataFrame(store.fetch_all(
    'rollouts', '*', configure=lambda query: query.eq(
        'experiment', SOURCE_SUFFIX_SENSITIVITY_EXPERIMENT),
    order_by=('rollout_id',)))
assert len(rows), f'No rows found for {SOURCE_SUFFIX_SENSITIVITY_EXPERIMENT}'
arms = validate_cohort(
    rows, expected_identities=EXPECTED_IDENTITIES,
    require_complete=REQUIRE_FULL_COHORT)
coverage = pd.DataFrame([{
    'arm': method, 'completed_rows': len(frame),
    'identities': len(frame[DIVERSITY_PAIR_KEYS].drop_duplicates()),
    'suites': frame.suite.nunique(),
} for method, frame in arms.items()])
print('Exact cohort validation passed')
display(coverage)

full_pairs = pair_arms(
    arms[Method.SUFFIX_SENSITIVITY], arms[Method.REFINEMENT])
tapered_pairs = pair_arms(
    arms[Method.SUFFIX_SENSITIVITY], arms[Method.TAPERED_REFINEMENT])
assert len(full_pairs) == len(tapered_pairs) == EXPECTED_IDENTITIES

## 3. Decode the diagnostic artifacts

This downloads only the diagnostic baseline's lightweight artifacts. They contain the 50-position uncertainty profile and tail-resampling response, not the full a-hat stacks. Set `REFRESH_ARTIFACTS=True` only when the stored rows have changed.

In [ ]:
cache_paths = {
    'features': OUTPUT / 'diagnostic_episode_features.csv',
    'records': OUTPUT / 'diagnostic_probe_records.csv',
    'uncertainty': OUTPUT / 'uncertainty_position_profile.csv',
    'sensitivity': OUTPUT / 'tail_sensitivity_position_profile.csv',
}
if not REFRESH_ARTIFACTS and all(path.exists() for path in cache_paths.values()):
    features = pd.read_csv(cache_paths['features'])
    probe_records = pd.read_csv(cache_paths['records'])
    uncertainty_profile = pd.read_csv(cache_paths['uncertainty'])
    sensitivity_profile = pd.read_csv(cache_paths['sensitivity'])
    print('Loaded cached artifact tables')
else:
    features, probe_records, uncertainty_profile, sensitivity_profile = (
        load_artifact_tables(
            store, arms[Method.SUFFIX_SENSITIVITY], progress=tqdm))
    features.to_csv(cache_paths['features'], index=False)
    probe_records.to_csv(cache_paths['records'], index=False)
    uncertainty_profile.to_csv(cache_paths['uncertainty'], index=False)
    sensitivity_profile.to_csv(cache_paths['sensitivity'], index=False)
    print('Downloaded and cached diagnostic artifacts')
assert len(features) == EXPECTED_IDENTITIES
assert features.rollout_id.nunique() == EXPECTED_IDENTITIES
print({'episode_features': len(features), 'probe_records': len(probe_records),
       'uncertainty_position_rows': len(uncertainty_profile),
       'tail_sensitivity_position_rows': len(sensitivity_profile)})

feature_payload = features.drop(columns=['suite', 'success']).rename(
    columns={'rollout_id': 'baseline_rollout_id'})
full_pairs = full_pairs.merge(
    feature_payload, on='baseline_rollout_id', validate='one_to_one')
tapered_pairs = tapered_pairs.merge(
    feature_payload, on='baseline_rollout_id', validate='one_to_one')

In [ ]:
required_features = [
    'u_first10_episode', 'u_first20_episode', 'u_full50_episode',
    'u_first10_first_chunk', 'u_first20_first_chunk', 'u_full50_first_chunk',
    'tail_to_prefix_l2_episode', 'tail_to_prefix_l2_first_chunk']
assert not features[required_features].isna().any().any(), (
    'Required diagnostic features contain missing values')
logged_u = arms[Method.SUFFIX_SENSITIVITY][
    ['rollout_id', 'u_mean_episode']].copy()
check = features[['rollout_id', 'u_full50_episode']].merge(
    logged_u, on='rollout_id', validate='one_to_one')
max_u_difference = (check.u_full50_episode - check.u_mean_episode).abs().max()
assert max_u_difference < 1e-5, (
    f'Artifact U50 disagrees with logged episode uncertainty by {max_u_difference}')
print(f'Artifact sanity check passed; max |U50 - logged U| = {max_u_difference:.2e}')

## 4. Primary paired success-rate result

These are causal rollout comparisons. `condition_minus_baseline_pp` is the SR difference over all 220 matched identities.

In [ ]:
arm_overall, arm_by_suite = summarize_all_arms(arms)
full_overall, full_by_suite = summarize_pair(full_pairs)
tapered_overall, tapered_by_suite = summarize_pair(tapered_pairs)
full_overall.insert(0, 'comparison', 'ordinary full P&P minus diagnostic baseline')
tapered_overall.insert(0, 'comparison', 'tapered P&P minus diagnostic baseline')
paired_overall = pd.concat([full_overall, tapered_overall], ignore_index=True)
print('Raw success rates')
display(arm_overall)
print('Paired changes; denominator is all 220 identities')
display(paired_overall[[
    'comparison', 'episodes', 'baseline_sr_pct', 'condition_sr_pct',
    'condition_minus_baseline_pp', 'delta_ci_low_pp', 'delta_ci_high_pp',
    'failure_to_success', 'success_to_failure', 'paired_p_value']])

full_vs_tapered = pair_arms(
    arms[Method.REFINEMENT], arms[Method.TAPERED_REFINEMENT])
full_vs_tapered_overall, _ = summarize_pair(full_vs_tapered)
print('Direct comparison: baseline column below is ordinary full P&P; condition is tapered P&P')
display(full_vs_tapered_overall)
arm_overall.to_csv(OUTPUT / 'arm_success_overall.csv', index=False)
paired_overall.to_csv(OUTPUT / 'paired_success_overall.csv', index=False)

In [ ]:
suite_pivot = arm_by_suite.pivot(
    index='suite', columns='arm', values='success_rate_pct').reset_index()
full_delta = full_by_suite[['suite', 'condition_minus_baseline_pp']].rename(
    columns={'condition_minus_baseline_pp': 'full_pnp_delta_pp'})
tapered_delta = tapered_by_suite[['suite', 'condition_minus_baseline_pp']].rename(
    columns={'condition_minus_baseline_pp': 'tapered_pnp_delta_pp'})
suite_plot = suite_pivot.merge(full_delta, on='suite').merge(tapered_delta, on='suite')
labels = suite_plot.suite.str.removeprefix('libero_')
x = np.arange(len(suite_plot)); width = .25
fig, axes = plt.subplots(2, 1, figsize=(15, 10), constrained_layout=True)
axes[0].bar(x - width, suite_plot.diagnostic_baseline, width,
            label='diagnostic baseline', color='#4C78A8')
axes[0].bar(x, suite_plot.full_pnp, width, label='ordinary full P&P', color='#F58518')
axes[0].bar(x + width, suite_plot.tapered_pnp, width,
            label='decaying-tail P&P', color='#54A24B')
axes[0].set_xticks(x, labels, rotation=40, ha='right')
axes[0].set(ylabel='Success rate (%)', ylim=(0, 105),
            title='Matched LIBERO-PRO success rates by suite')
axes[0].legend(); axes[0].grid(axis='y', alpha=.2)
axes[1].bar(x - width/2, suite_plot.full_pnp_delta_pp, width,
            label='ordinary full P&P minus baseline', color='#F58518')
axes[1].bar(x + width/2, suite_plot.tapered_pnp_delta_pp, width,
            label='decaying-tail P&P minus baseline', color='#54A24B')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].axhline(full_overall.condition_minus_baseline_pp.iloc[0],
                color='#F58518', linestyle='--', alpha=.7)
axes[1].axhline(tapered_overall.condition_minus_baseline_pp.iloc[0],
                color='#54A24B', linestyle='--', alpha=.7)
axes[1].set_xticks(x, labels, rotation=40, ha='right')
axes[1].set(ylabel='SR change over all matched suite episodes (pp)',
            title='Paired whole-cohort change by suite')
axes[1].legend(); axes[1].grid(axis='y', alpha=.2)
fig.savefig(OUTPUT / 'paired_success_by_suite.png', dpi=180, bbox_inches='tight')
plt.show()
display(suite_plot)

## 5. Uncertainty as a failure detector

`first10`, `first20`, and `full50` refer to positions within each generated 50-action chunk. The episode scores average those measurements over all prediction chunks and both probed Euler steps. The first-chunk variants use only the initial observation. AUC 0.5 is chance; above 0.5 means larger uncertainty predicts failure.

In [ ]:
uncertainty_scores = [
    'u_first10_episode', 'u_first20_episode', 'u_full50_episode',
    'u_first10_first_chunk', 'u_first20_first_chunk', 'u_full50_first_chunk']
auc_table = failure_auc_table(features, uncertainty_scores, n_boot=2000)
pooled_auc = auc_table[auc_table.suite.eq('pooled')].copy()
print('Pooled failure AUC')
display(pooled_auc[[
    'score_name', 'episodes', 'failures', 'failure_auc',
    'auc_ci_low', 'auc_ci_high']])
print('Per-suite failure AUC')
display(auc_table[~auc_table.suite.eq('pooled')])
auc_table.to_csv(OUTPUT / 'uncertainty_failure_auc.csv', index=False)

primary_scores = ['u_first10_episode', 'u_first20_episode', 'u_full50_episode']
primary_labels = {'u_first10_episode': 'first 10 actions',
                  'u_first20_episode': 'first 20 actions',
                  'u_full50_episode': 'full 50 actions'}
plot_auc = auc_table[
    ~auc_table.suite.eq('pooled') & auc_table.score_name.isin(primary_scores)
    & auc_table.failure_auc.notna()].copy()
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
for index, score in enumerate(primary_scores):
    group = plot_auc[plot_auc.score_name.eq(score)].sort_values('suite')
    y = np.arange(len(group)) + (index - 1) * .18
    axes[0].errorbar(
        group.failure_auc, y,
        xerr=np.vstack((group.failure_auc - group.auc_ci_low,
                        group.auc_ci_high - group.failure_auc)),
        fmt='o', capsize=2, label=primary_labels[score])
base_group = plot_auc[plot_auc.score_name.eq(primary_scores[0])].sort_values('suite')
axes[0].set_yticks(np.arange(len(base_group)),
                   base_group.suite.str.removeprefix('libero_'))
axes[0].axvline(.5, color='black', linestyle='--', linewidth=1)
axes[0].set(xlim=(0, 1), xlabel='Failure ROC-AUC (95% bootstrap CI)',
            title='Uncertainty failure AUC by suite')
axes[0].legend(fontsize=8); axes[0].grid(axis='x', alpha=.2)

from sklearn.metrics import roc_curve
failures = (~features.success.astype(bool)).astype(int).to_numpy()
for score in primary_scores:
    fpr, tpr, _ = roc_curve(failures, features[score].to_numpy(float))
    auc = pooled_auc[pooled_auc.score_name.eq(score)].failure_auc.iloc[0]
    axes[1].plot(fpr, tpr, label=f'{primary_labels[score]}: AUC {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', label='chance')
axes[1].set(xlabel='False-positive rate', ylabel='True-positive rate',
            title='Pooled failure ROC')
axes[1].legend(); axes[1].grid(alpha=.2)
fig.savefig(OUTPUT / 'uncertainty_failure_auc_and_roc.png',
            dpi=180, bbox_inches='tight')
plt.show()

## 6. Where uncertainty lies inside the 50-action prediction

This plot asks whether failures disproportionately increase uncertainty in the executed prefix or in the unused tail.

In [ ]:
u_position = (uncertainty_profile
    .groupby(['success', 'action_position']).uncertainty
    .agg(['mean', 'std', 'count']).reset_index())
u_position['sem'] = u_position['std'] / np.sqrt(u_position['count'])
fig, ax = plt.subplots(figsize=(11, 5))
for success, group in u_position.groupby('success', sort=False):
    label = 'successful baseline episodes' if bool(success) else 'failed baseline episodes'
    color = '#4C78A8' if bool(success) else '#E45756'
    ax.plot(group.action_position, group['mean'], label=label, color=color)
    ax.fill_between(group.action_position, group['mean'] - group['sem'],
                    group['mean'] + group['sem'], color=color, alpha=.16)
ax.axvline(9.5, color='black', linestyle='--', label='executed-prefix boundary')
ax.axvline(19.5, color='#9467BD', linestyle=':', label='taper reaches zero at position 20')
ax.set(xlabel='Action position within generated 50-action chunk',
       ylabel='Mean P&P uncertainty',
       title='Per-position uncertainty profile by baseline outcome')
ax.legend(); ax.grid(alpha=.2)
fig.tight_layout(); fig.savefig(OUTPUT / 'uncertainty_action_position_profile.png', dpi=180)
plt.show()

## 7. Exploratory uncertainty-window sweeps

For each cell below, all 220 episodes remain in the denominator. The policy uses the matching P&P outcome when baseline uncertainty falls inside the window and otherwise keeps the diagnostic baseline outcome. These are in-sample exploratory results, exactly like the earlier PRO window sweeps.

In [ ]:
comparisons = {'ordinary_full_pnp': full_pairs,
               'decaying_tail_pnp': tapered_pairs}
all_sweeps, all_best, all_thresholds = [], [], []
for comparison, pair in comparisons.items():
    for score_name in uncertainty_scores:
        sweep = window_sweep(
            pair, score_column=score_name, grid_size=GRID_SIZE,
            min_selected=MIN_SELECTED)
        sweep.insert(0, 'comparison', comparison)
        all_sweeps.append(sweep)
        best = top_windows(sweep, n=10)
        best.insert(0, 'comparison', comparison)
        all_best.append(best)
        thresholds = threshold_sweep(
            pair, score_column=score_name, grid_size=33,
            min_selected=MIN_SELECTED)
        thresholds.insert(0, 'comparison', comparison)
        all_thresholds.append(thresholds)
window_results = pd.concat(all_sweeps, ignore_index=True)
top_window_results = pd.concat(all_best, ignore_index=True)
threshold_results = pd.concat(all_thresholds, ignore_index=True)
best_per_score = (top_window_results.sort_values('rank')
                  .groupby(['comparison', 'score_name'], sort=False).head(1)
                  .reset_index(drop=True))
best_thresholds = (threshold_results[threshold_results.eligible]
    .sort_values(['comparison', 'score_name', 'delta_pp', 'episodes_refined'],
                 ascending=[True, True, False, False])
    .groupby(['comparison', 'score_name'], sort=False).head(1).reset_index(drop=True))

def readable_windows(frame):
    output = frame[[
        'comparison', 'score_name', 'rank', 'episodes_in_sr_denominator',
        'lower', 'upper', 'episodes_refined', 'baseline_sr',
        'window_policy_sr', 'delta_pp', 'selected_F_to_S', 'selected_S_to_F']].copy()
    output['baseline_sr_all_episodes_pct'] = 100 * output.pop('baseline_sr')
    output['window_policy_sr_all_episodes_pct'] = 100 * output.pop('window_policy_sr')
    return output.rename(columns={
        'score_name': 'uncertainty_measure',
        'delta_pp': 'window_policy_minus_baseline_pp'})

def readable_thresholds(frame):
    output = frame[[
        'comparison', 'score_name', 'episodes_in_sr_denominator', 'threshold',
        'episodes_refined', 'baseline_sr', 'threshold_policy_sr', 'delta_pp',
        'selected_F_to_S', 'selected_S_to_F']].copy()
    output['baseline_sr_all_episodes_pct'] = 100 * output.pop('baseline_sr')
    output['threshold_policy_sr_all_episodes_pct'] = 100 * output.pop('threshold_policy_sr')
    return output.rename(columns={
        'score_name': 'uncertainty_measure',
        'delta_pp': 'threshold_policy_minus_baseline_pp'})

print('Best exploratory bounded window for every arm and uncertainty score')
display(readable_windows(best_per_score))
print('Best exploratory U >= threshold rule for every arm and uncertainty score')
display(readable_thresholds(best_thresholds))
print('Top 5 bounded windows for each arm and score (full top-10 table is saved to CSV)')
top_five = (top_window_results[top_window_results['rank'] <= 5]
            .sort_values(['comparison', 'score_name', 'rank']))
display(readable_windows(top_five))
window_results.to_csv(OUTPUT / 'uncertainty_window_sweep.csv', index=False)
top_window_results.to_csv(OUTPUT / 'uncertainty_top_windows.csv', index=False)
threshold_results.to_csv(OUTPUT / 'uncertainty_threshold_sweep.csv', index=False)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10), constrained_layout=True)
for row_index, (comparison, _) in enumerate(comparisons.items()):
    pretty_comparison = comparison.replace('_', ' ')
    for column_index, score_name in enumerate(primary_scores):
        sweep = window_results[
            window_results.comparison.eq(comparison)
            & window_results.score_name.eq(score_name)]
        pivot = sweep.pivot(index='lower', columns='upper', values='delta_pp').sort_index()
        limit = max(1, np.nanmax(np.abs(pivot.to_numpy())))
        image = axes[row_index, column_index].imshow(
            pivot.to_numpy(), aspect='auto', origin='lower',
            cmap='RdYlGn', vmin=-limit, vmax=limit)
        y_idx = np.linspace(0, len(pivot.index) - 1, 6).astype(int)
        x_idx = np.linspace(0, len(pivot.columns) - 1, 7).astype(int)
        axes[row_index, column_index].set_yticks(
            y_idx, [f'{pivot.index[i]:.3f}' for i in y_idx])
        axes[row_index, column_index].set_xticks(
            x_idx, [f'{pivot.columns[i]:.3f}' for i in x_idx], rotation=35)
        axes[row_index, column_index].set(
            xlabel='Upper uncertainty bound', ylabel='Lower uncertainty bound',
            title=f'{pretty_comparison}\n{primary_labels[score_name]}')
        fig.colorbar(image, ax=axes[row_index, column_index],
                     label='Whole-cohort SR change (pp)')
fig.suptitle('Exploratory bounded-window sweeps (all 220 episodes in every denominator)',
             fontsize=15)
fig.savefig(OUTPUT / 'uncertainty_window_sweep_heatmaps.png',
            dpi=180, bbox_inches='tight')
plt.show()

score_labels = {
    'u_first10_episode': 'U10, full episode',
    'u_first20_episode': 'U20, full episode',
    'u_full50_episode': 'U50, full episode',
    'u_first10_first_chunk': 'U10, first chunk',
    'u_first20_first_chunk': 'U20, first chunk',
    'u_full50_first_chunk': 'U50, first chunk'}
fig, ax = plt.subplots(figsize=(12, 5))
score_order = uncertainty_scores
x = np.arange(len(score_order)); width = .36
for offset, comparison, color in [
        (-width/2, 'ordinary_full_pnp', '#F58518'),
        (width/2, 'decaying_tail_pnp', '#54A24B')]:
    values = (best_per_score[best_per_score.comparison.eq(comparison)]
              .set_index('score_name').reindex(score_order).delta_pp)
    ax.bar(x + offset, values, width, label=comparison.replace('_', ' '), color=color)
ax.axhline(0, color='black', linewidth=1)
ax.set_xticks(x, [score_labels[name] for name in score_order], rotation=25, ha='right')
ax.set(ylabel='Best exploratory whole-cohort SR change (pp)',
       title='Best window by action horizon and observation horizon')
ax.legend(); ax.grid(axis='y', alpha=.2)
fig.tight_layout(); fig.savefig(OUTPUT / 'best_window_by_uncertainty_score.png', dpi=180)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9), constrained_layout=True)
for row_index, comparison in enumerate(comparisons):
    pretty_comparison = comparison.replace('_', ' ')
    for column_index, score_name in enumerate(primary_scores):
        curve = threshold_results[
            threshold_results.comparison.eq(comparison)
            & threshold_results.score_name.eq(score_name)]
        axes[row_index, column_index].plot(
            curve.threshold, curve.delta_pp, marker='o', markersize=3)
        axes[row_index, column_index].axhline(0, color='black', linewidth=1)
        axes[row_index, column_index].set(
            xlabel='Refine when U >= threshold',
            ylabel='Whole-cohort SR change (pp)',
            title=f'{pretty_comparison}\n{primary_labels[score_name]}')
        axes[row_index, column_index].grid(alpha=.2)
fig.suptitle('One-sided uncertainty-threshold sweeps (all 220 episodes in every denominator)',
             fontsize=15)
fig.savefig(OUTPUT / 'uncertainty_threshold_sweep.png',
            dpi=180, bbox_inches='tight')
plt.show()

## 8. Success-rate graph for each arm's optimal full-episode window

This recreates the earlier optimal-window per-suite graph. The selected window is the best among U10, U20, and U50 episode-level scores—not the first-observation variants.

In [ ]:
best_episode_windows = (best_per_score[best_per_score.score_name.isin(primary_scores)]
    .sort_values(['comparison', 'delta_pp', 'episodes_refined'],
                 ascending=[True, False, False])
    .groupby('comparison', sort=False).head(1).reset_index(drop=True))
display(best_episode_windows[[
    'comparison', 'score_name', 'lower', 'upper', 'episodes_refined',
    'window_policy_sr', 'delta_pp']])
fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)
optimal_suite_tables = []
for row_index, (comparison, pair) in enumerate(comparisons.items()):
    best = best_episode_windows[best_episode_windows.comparison.eq(comparison)].iloc[0]
    suite = apply_window_by_suite(
        pair, score_column=best.score_name, lower=best.lower, upper=best.upper)
    suite.insert(0, 'comparison', comparison)
    suite['score_name'] = best.score_name
    optimal_suite_tables.append(suite)
    labels = suite.suite.str.removeprefix('libero_'); x = np.arange(len(suite)); width = .38
    axes[row_index, 0].bar(x - width/2, suite.baseline_sr_pct, width,
                           label='diagnostic baseline', color='#4C78A8')
    color = '#F58518' if comparison == 'ordinary_full_pnp' else '#54A24B'
    axes[row_index, 0].bar(x + width/2, suite.window_policy_sr_pct, width,
                           label='optimal window policy', color=color)
    axes[row_index, 0].set_xticks(x, labels, rotation=40, ha='right')
    pretty_comparison = comparison.replace('_', ' ')
    axes[row_index, 0].set(ylabel='Success rate (%)', ylim=(0, 105),
        title=f'{pretty_comparison}: {best.score_name}\nwindow [{best.lower:.4f}, {best.upper:.4f}]')
    axes[row_index, 0].legend(); axes[row_index, 0].grid(axis='y', alpha=.2)
    colors = np.where(suite.window_minus_baseline_pp >= 0, '#54A24B', '#E45756')
    axes[row_index, 1].bar(x, suite.window_minus_baseline_pp, color=colors)
    axes[row_index, 1].axhline(0, color='black', linewidth=1)
    axes[row_index, 1].axhline(best.delta_pp, color='#9467BD', linestyle='--',
        label=f'overall: {best.delta_pp:+.2f} pp')
    axes[row_index, 1].set_xticks(x, labels, rotation=40, ha='right')
    axes[row_index, 1].set(
        ylabel='Window policy minus baseline SR (pp)',
        title='Whole-matched-cohort change by suite')
    axes[row_index, 1].legend(); axes[row_index, 1].grid(axis='y', alpha=.2)
fig.savefig(OUTPUT / 'optimal_window_success_by_suite.png', dpi=180, bbox_inches='tight')
plt.show()
optimal_suite_tables = pd.concat(optimal_suite_tables, ignore_index=True)
optimal_suite_tables.to_csv(OUTPUT / 'optimal_window_by_suite.csv', index=False)
display(optimal_suite_tables)

## 9. Does tail-to-prefix influence predict failure or refinement benefit?

Tail sensitivity is measured without changing the executed baseline trajectory: positions 0–9 are held fixed while positions 10–49 are independently resampled. Higher L2 means the predicted executed prefix changed more. The quartile SR deltas below are conditional within each quartile; unlike the primary result, they are not whole-cohort deltas.

In [ ]:
sensitivity_scores = [
    'tail_to_prefix_l2_episode', 'tail_to_prefix_abs_episode',
    'tail_to_prefix_std_episode', 'tail_gripper_flip_episode',
    'tail_to_prefix_l2_first_chunk']
sensitivity_auc = failure_auc_table(
    features, sensitivity_scores, by_suite=False, n_boot=3000)
print('Can tail sensitivity identify baseline failures?')
display(sensitivity_auc[[
    'score_name', 'episodes', 'failures', 'failure_auc',
    'auc_ci_low', 'auc_ci_high']])

correction_rows = []
for comparison, pair in comparisons.items():
    failed = pair[~pair.baseline_success].copy()
    for score_name in ('tail_to_prefix_l2_episode',
                       'tail_to_prefix_l2_first_chunk'):
        auc, low, high = bootstrap_rank_auc(
            failed.condition_success, failed[score_name], n_boot=3000)
        correction_rows.append({
            'comparison': comparison, 'score_name': score_name,
            'baseline_failures': len(failed),
            'corrected_failures': int(failed.condition_success.sum()),
            'correction_auc': auc, 'auc_ci_low': low, 'auc_ci_high': high})
correction_auc = pd.DataFrame(correction_rows)
print('Among baseline failures, can tail sensitivity identify F->S corrections?')
display(correction_auc)

quartiles = sensitivity_quartiles(full_pairs, tapered_pairs)
print('Outcome rates within baseline tail-sensitivity quartiles')
display(quartiles)
sensitivity_auc.to_csv(OUTPUT / 'tail_sensitivity_failure_auc.csv', index=False)
correction_auc.to_csv(OUTPUT / 'tail_sensitivity_correction_auc.csv', index=False)
quartiles.to_csv(OUTPUT / 'tail_sensitivity_quartiles.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)
# Prefix-position sensitivity profile.
sens_position = (sensitivity_profile
    .groupby(['success', 'executed_action_position']).tail_to_prefix_l2
    .agg(['mean', 'std', 'count']).reset_index())
sens_position['sem'] = sens_position['std'] / np.sqrt(sens_position['count'])
for success, group in sens_position.groupby('success', sort=False):
    label = 'baseline success' if bool(success) else 'baseline failure'
    color = '#4C78A8' if bool(success) else '#E45756'
    axes[0].plot(group.executed_action_position, group['mean'], marker='o',
                 color=color, label=label)
    axes[0].fill_between(group.executed_action_position,
        group['mean'] - group['sem'], group['mean'] + group['sem'],
        color=color, alpha=.15)
axes[0].set(xticks=np.arange(10), xlabel='Executed-prefix action position',
    ylabel='Tail-induced prediction change (L2)',
    title='Which executed actions depend on the tail?')
axes[0].legend(); axes[0].grid(alpha=.2)

# Direct correlation-style risk curve.
risk = features[['tail_to_prefix_l2_episode', 'success']].copy()
risk['bin'] = pd.qcut(risk.tail_to_prefix_l2_episode.rank(method='first'), 5, labels=False) + 1
risk_curve = risk.groupby('bin').agg(
    sensitivity_mean=('tail_to_prefix_l2_episode', 'mean'),
    episodes=('success', 'size'), failure_rate=('success', lambda value: 1 - value.mean())
).reset_index()
error = np.sqrt(risk_curve.failure_rate * (1 - risk_curve.failure_rate)
                / risk_curve.episodes)
axes[1].errorbar(risk_curve.sensitivity_mean, 100 * risk_curve.failure_rate,
                 yerr=100 * error, marker='o', capsize=3, color='#9467BD')
primary_auc = sensitivity_auc[
    sensitivity_auc.score_name.eq('tail_to_prefix_l2_episode')].failure_auc.iloc[0]
axes[1].set(xlabel='Mean tail-to-prefix L2 in sensitivity quintile',
    ylabel='Baseline failure rate (%)',
    title=f'Tail influence vs failure rate (AUC {primary_auc:.3f})')
axes[1].grid(alpha=.2)

# Refinement effects within the same sensitivity strata.
x = quartiles.sensitivity_quartile.to_numpy(); width = .35
axes[2].bar(x - width/2, quartiles.full_minus_baseline_pp_within_quartile, width,
            label='ordinary full P&P', color='#F58518')
axes[2].bar(x + width/2, quartiles.tapered_minus_baseline_pp_within_quartile, width,
            label='decaying-tail P&P', color='#54A24B')
axes[2].axhline(0, color='black', linewidth=1)
axes[2].set(xticks=x, xlabel='Baseline tail-sensitivity quartile',
    ylabel='Paired SR change within quartile (pp)',
    title='Who benefits from each refinement?')
axes[2].legend(); axes[2].grid(axis='y', alpha=.2)
fig.savefig(OUTPUT / 'tail_sensitivity_analysis.png', dpi=180, bbox_inches='tight')
plt.show()

## 10. Concise result

In [ ]:
for _, row in paired_overall.iterrows():
    print(f"{row.comparison}: {row.condition_sr_pct:.2f}% vs "
          f"{row.baseline_sr_pct:.2f}% = {row.condition_minus_baseline_pp:+.2f} pp "
          f"(95% CI {row.delta_ci_low_pp:+.2f} to {row.delta_ci_high_pp:+.2f}); "
          f"{int(row.failure_to_success)} F->S, {int(row.success_to_failure)} S->F")
print('\nBest full-episode uncertainty windows (exploratory):')
for _, row in best_episode_windows.iterrows():
    print(f"{row.comparison}, {row.score_name}: [{row.lower:.4f}, {row.upper:.4f}], "
          f"refine {int(row.episodes_refined)}/{int(row.episodes_in_sr_denominator)}, "
          f"whole-cohort delta {row.delta_pp:+.2f} pp")
print('\nPrimary tail-sensitivity failure AUC:', f'{primary_auc:.3f}')
print('All tables and figures saved to:', OUTPUT.resolve())